##### I created a scaled analytical workload from the same distribution and schema, so that DuckDB’s storage and execution behavior becomes easier to observe.

In [1]:
import pandas as pd

# Load the cleaned Superstore dataset
df = pd.read_csv("cleaned superstore dataset.csv")

# Scale the dataset by duplicating rows 20 times
df_scaled = pd.concat([df] * 20, ignore_index=True)

# Save the scaled dataset
df_scaled.to_csv("superstore_scaled.csv", index=False)

print("Original shape:", df.shape)
print("Scaled shape:", df_scaled.shape)
print("Scaled dataset saved as superstore_scaled.csv")

Original shape: (9993, 21)
Scaled shape: (199860, 21)
Scaled dataset saved as superstore_scaled.csv


##### An in-memory database instance was created to enable rapid experimentation and query execution. This setup allows direct interaction with the DuckDB engine and supports efficient execution of analytical queries without requiring a separate database server.

In [2]:
import duckdb
con = duckdb.connect(database=':memory:') ###in-memory database for Phase 2 

In [3]:
# Load the scaled dataset into DuckDB
con.execute("""
CREATE TABLE sales_data AS
SELECT * FROM 'superstore_scaled.csv'
""")

In [4]:
# to verify the table
con.execute("SELECT COUNT(*) FROM sales_data").fetchall()

[(199860,)]

In [5]:
# to preview data
con.execute("SELECT * FROM sales_data LIMIT 5").fetchdf()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit Margin %
0,CA-2019-103800,2019-01-03,2019-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,...,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,33.75
1,CA-2019-112326,2019-01-04,2019-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,36.25
2,CA-2019-112326,2019-01-04,2019-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,-23.75
3,CA-2019-112326,2019-01-04,2019-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,-155.00
4,CA-2019-141817,2019-01-05,2019-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,...,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,25.00


# Project Focus (internals)

## Query Execution

### Query 1--> Sales by region

In [6]:
q1 = """
SELECT Region, SUM(Sales) AS total_sales
FROM sales_data
GROUP BY Region
ORDER BY total_sales DESC
"""

con.execute(q1).fetchdf()

,Region,total_sales
0,West,1.450916e+07
1,East,1.357000e+07
2,Central,1.002480e+07
3,South,7.834438e+06


In [7]:
print(con.execute("EXPLAIN ANALYZE " + q1).fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT Region, SUM(Sales) AS total_sales FROM sales_data GROUP BY Region ORDER BY total_sales DESC 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0209s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│       

### Query 2 --> Filtered regional aggregation

In [8]:
q2 = """
SELECT Region, SUM(Sales) AS total_sales
FROM sales_data
WHERE Category = 'Technology'
GROUP BY Region
ORDER BY total_sales DESC
"""

con.execute(q2).fetchdf()

,Region,total_sales
0,East,5299479.62
1,West,5039836.64
2,Central,3408326.24
3,South,2975438.16


In [9]:
print(con.execute("EXPLAIN ANALYZE " + q2).fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT Region, SUM(Sales) AS total_sales FROM sales_data WHERE Category = 'Technology' GROUP BY Region ORDER BY total_sales DESC 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0174s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__inter

### Query 3 --> Monthly sales trend

In [10]:
q3 = """
SELECT strftime(CAST("Order Date" AS DATE), '%Y-%m') AS month,
       SUM(Sales) AS total_sales
FROM sales_data
GROUP BY month
ORDER BY month
"""

con.execute(q3).fetchdf()

,month,total_sales
0,2019-01,284737.900
1,2019-02,90397.840
2,2019-03,1113820.180
3,2019-04,560279.460
4,2019-05,472965.740
5,2019-06,691902.552
6,2019-07,678927.860
7,2019-08,558189.370
8,2019-09,1635547.016
9,2019-10,629067.860


In [11]:
print(con.execute("EXPLAIN ANALYZE " + q3).fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT strftime(CAST("Order Date" AS DATE), '%Y-%m') AS month,        SUM(Sales) AS total_sales FROM sales_data GROUP BY month ORDER BY month 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0348s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────

### Query 4 --> Sales and profit by category

In [12]:
q4 = """
SELECT Category,
       SUM(Sales) AS total_sales,
       SUM(Profit) AS total_profit
FROM sales_data
GROUP BY Category
ORDER BY total_sales DESC
"""

con.execute(q4).fetchdf()

,Category,total_sales,total_profit
0,Technology,1.672308e+07,2909098.962
1,Furniture,1.483437e+07,369266.632
2,Office Supplies,1.438094e+07,2449816.016


In [13]:
print(con.execute("EXPLAIN ANALYZE " + q4).fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT Category,        SUM(Sales) AS total_sales,        SUM(Profit) AS total_profit FROM sales_data GROUP BY Category ORDER BY total_sales DESC 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0664s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────

## Query Summary 

| Query                          | Columns Used              | Filter | Groups Returned | Total Time | Key Observation              |
|--------------------------------|---------------------------|--------|-----------------|------------|------------------------------|
| Q1 Sales by Region             | Region, Sales             | No     | 4               | 0.0209s    | Simple baseline aggregation  |
| Q2 Technology Sales by Region  | Category, Region, Sales   | Yes    | 4               | 0.0174s    | Filtered aggregation         |
| Q3 Monthly Sales Trend         | Order Date, Sales         | No     | 48              | 0.0348s    | Derived time grouping        |
| Q4 Sales and Profit by Category| Category, Sales, Profit   | No     | 3               | 0.0664s    | Two aggregate measures       |

# Phase 3

## Experiment Summary

| Query              | Columns Used               | Rows Processed | Key Operators                               | Execution Time |
|--------------------|----------------------------|----------------|----------------------------------------------|----------------|
| Q1: Sales by Region| Region, Sales              | ~199,860       | TABLE_SCAN, HASH_GROUP_BY                    | ~0.0209s       |
| Q2: Filtered Sales | Category, Region, Sales    | ~36,940        | TABLE_SCAN (filter), HASH_GROUP_BY           | ~0.0174s       |
| Q3: Monthly Sales  | Order Date, Sales          | ~199,860       | TABLE_SCAN, HASH_GROUP_BY, transformation    | ~0.0348s       |
| Q4: Sales & Profit | Category, Sales, Profit    | ~199,860       | TABLE_SCAN, HASH_GROUP_BY                    | ~0.0664s       |

### Observations

- Filtering (Q2) reduces the number of rows processed before aggregation.
- Time-based queries (Q3) introduce additional computation, increasing execution time.
- Queries with additional computations or aggregates (Q4) increase execution time, even when the execution structure remains the same